# 07 - Article Type and Method Labels

This notebook classifies article methods with the Codex CLI instead of keyword rules. It uses the hand-sketched taxonomy: empirical articles split into quantitative or qualitative and then historical or contemporary; theoretical articles split into descriptive or normative. The default run mode is a small sample so the prompt and outputs can be checked before running the full corpus.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
import json
import math
import subprocess
import tempfile
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# -----------------------------
# Run configuration
# -----------------------------
CODEX_MODEL = "gpt-5.5"
CODEX_REASONING_EFFORT = "medium"

# Start with RUN_SCOPE = "sample". Change to "full" after checking the sample output.
RUN_SCOPE = "sample"  # "sample" or "full"
SAMPLE_N = 30
SAMPLE_RANDOM_STATE = 42
BATCH_SIZE = 8
MAX_TEXT_CHARS_PER_FIELD = 2500
CODEX_TIMEOUT_SECONDS = 900
MAX_RETRIES = 2
SLEEP_BETWEEN_CALLS = 1.0
RESUME_FROM_CACHE = True

# In sample mode the notebook writes sample-suffixed outputs so it does not overwrite full-run tables.
WRITE_COMPATIBILITY_OUTPUTS = RUN_SCOPE == "full"
OUTPUT_SUFFIX = "sample" if RUN_SCOPE == "sample" else "full"

LABELS_PATH = ROOT / f"outputs/tables/article_method_labels_codex_{OUTPUT_SUFFIX}.csv"
VALIDATION_PATH = ROOT / f"outputs/labels/article_method_validation_sample_{OUTPUT_SUFFIX}.csv"
VALIDATION_RESULTS_PATH = ROOT / f"outputs/tables/article_method_validation_results_{OUTPUT_SUFFIX}.csv"
FIGURE_PATH = ROOT / f"outputs/figures/fig_17_methodological_category_over_time_{OUTPUT_SUFFIX}.png"
CACHE_PATH = ROOT / "outputs/cache/codex_method_classifications.jsonl"
RAW_RESPONSE_DIR = ROOT / "outputs/diagnostics/codex_method_classifier_raw"

LEAF_CATEGORIES = [
    "empirical_quantitative_historical",
    "empirical_quantitative_contemporary",
    "empirical_qualitative_historical",
    "empirical_qualitative_contemporary",
    "theoretical_descriptive",
    "theoretical_normative",
    "unclear",
]

CODEX_CLASSIFICATION_PROMPT = """
You are classifying journal article records from Historical Social Research by methodological type.
Use this taxonomy, based on the supplied hand-drawn scheme:

Article
- empirical
  - quantitative
    - historical
    - contemporary
  - qualitative
    - historical
    - contemporary
- theoretical
  - descriptive
  - normative

Return one primary leaf category for each article:
- empirical_quantitative_historical
- empirical_quantitative_contemporary
- empirical_qualitative_historical
- empirical_qualitative_contemporary
- theoretical_descriptive
- theoretical_normative
- unclear

Definitions:
- Empirical means the article analyzes observations, cases, documents, sources, interviews, surveys, datasets, measurements, events, populations, institutions, or other evidence from the world.
- Quantitative empirical means numerical, statistical, measurement-based, formal data analysis, survey statistics, coding/counting, regression, sequence/panel/time-series analysis, network metrics, GIS/spatial measurement, or comparable quantitative evidence.
- Qualitative empirical means interpretive source analysis, archival reading, case studies, discourse/content interpretation, interviews, ethnography, qualitative comparison, or historical narrative based mainly on non-numeric evidence.
- Historical empirical means the empirical object is mainly in the past or uses historical sources/data to study past processes, even if the article was written recently.
- Contemporary empirical means the empirical object is mainly current/recent society, present-day institutions, or contemporary data at the time of publication.
- Theoretical descriptive means conceptual, theoretical, historiographical, methodological, or explanatory argument that primarily describes, clarifies, synthesizes, models, or interprets without making an explicit normative/prescriptive claim.
- Theoretical normative means the article primarily argues about what should be done, what is justified, ethical/political/legal evaluation, critique of norms, or prescriptive recommendations.

Decision rules:
- Choose the best single leaf category. If an article mixes methods, choose the dominant one and mention the mixture in notes.
- Prefer title, abstract, keywords, and article type. Use topic labels only as weak context when the article text is sparse.
- Do not infer quantitative just because words like analysis, data, or model appear. Look for actual numerical/statistical/measurement work.
- Do not infer normative just because an article is critical. Use normative only when the main contribution evaluates, prescribes, or argues about what ought to be done.
- If the supplied metadata is too thin or genuinely ambiguous, use unclear and set confidence no higher than 0.35.
- Keep evidence short: cite the phrase or metadata feature that drove the decision, not a long explanation.
- Return JSON only, matching the schema. Do not include markdown.
""".strip()

CODEX_OUTPUT_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["classifications"],
    "properties": {
        "classifications": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "article_id",
                    "empirical_or_theoretical",
                    "empirical_method",
                    "temporal_scope",
                    "theoretical_type",
                    "leaf_category",
                    "confidence",
                    "evidence",
                    "notes",
                ],
                "properties": {
                    "article_id": {"type": "string"},
                    "empirical_or_theoretical": {"type": "string", "enum": ["empirical", "theoretical", "unclear"]},
                    "empirical_method": {"type": "string", "enum": ["quantitative", "qualitative", "not_applicable", "unclear"]},
                    "temporal_scope": {"type": "string", "enum": ["historical", "contemporary", "not_applicable", "unclear"]},
                    "theoretical_type": {"type": "string", "enum": ["descriptive", "normative", "not_applicable", "unclear"]},
                    "leaf_category": {"type": "string", "enum": LEAF_CATEGORIES},
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                    "evidence": {"type": "string"},
                    "notes": {"type": "string"},
                },
            },
        }
    },
}

print(f"Codex model: {CODEX_MODEL}, reasoning effort: {CODEX_REASONING_EFFORT}")
print(f"Run scope: {RUN_SCOPE}, sample N: {SAMPLE_N if RUN_SCOPE == 'sample' else 'all'}")
print(f"Labels output: {LABELS_PATH}")


In [ ]:
articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
article_topics = pd.read_csv(ROOT / "outputs/tables/article_topics.csv")

df = articles.merge(
    article_topics[["article_id", "topic_id", "topic_label_human", "topic_macro_category"]],
    on="article_id",
    how="left",
)

METADATA_COLUMNS = [
    "article_id",
    "year",
    "period",
    "language",
    "type_en",
    "topic_id",
    "topic_label_human",
    "topic_macro_category",
]

TEXT_COLUMNS = ["title", "abstract_en", "abstract_de", "keywords_en", "keywords_de"]


def clean_value(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return compact_ws(str(value))


def truncate_text(value, max_chars=MAX_TEXT_CHARS_PER_FIELD):
    value = clean_value(value)
    if len(value) <= max_chars:
        return value
    return value[:max_chars].rsplit(" ", 1)[0] + " [...]"


def make_record(row):
    record = {
        "article_id": clean_value(row.get("article_id")),
        "year": int(row["year"]) if pd.notna(row.get("year")) else None,
        "language": clean_value(row.get("language")),
        "article_type": clean_value(row.get("type_en")),
        "topic_label": clean_value(row.get("topic_label_human")),
        "topic_macro_category": clean_value(row.get("topic_macro_category")),
    }
    for col in TEXT_COLUMNS:
        if col in row.index:
            record[col] = truncate_text(row.get(col))
    return record


def select_target_articles(source_df):
    if RUN_SCOPE == "full":
        return source_df.copy()
    if RUN_SCOPE != "sample":
        raise ValueError('RUN_SCOPE must be "sample" or "full"')

    n = min(SAMPLE_N, len(source_df))
    strata = source_df[["period", "language", "topic_macro_category"]].fillna("unknown").agg("|".join, axis=1)
    try:
        if strata.value_counts().min() >= 2 and strata.nunique() < n:
            sample, _ = train_test_split(source_df, train_size=n, random_state=SAMPLE_RANDOM_STATE, stratify=strata)
            return sample.sort_values(["period", "topic_macro_category", "year", "article_id"])
    except ValueError:
        pass
    return source_df.sample(n=n, random_state=SAMPLE_RANDOM_STATE).sort_values(["period", "topic_macro_category", "year", "article_id"])


def load_cache(path):
    if not RESUME_FROM_CACHE or not path.exists():
        return {}
    cached = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            cached[item["article_id"]] = item
    return cached


def append_cache(path, classifications):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for item in classifications:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def parse_json_object(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.removeprefix("json").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        return json.loads(text[start : end + 1])


def normalize_classification(item):
    leaf = clean_value(item.get("leaf_category"))
    if leaf not in LEAF_CATEGORIES:
        leaf = "unclear"

    if leaf.startswith("empirical_quantitative_"):
        family = "empirical"
        empirical_method = "quantitative"
        temporal_scope = leaf.removeprefix("empirical_quantitative_")
        theoretical_type = "not_applicable"
    elif leaf.startswith("empirical_qualitative_"):
        family = "empirical"
        empirical_method = "qualitative"
        temporal_scope = leaf.removeprefix("empirical_qualitative_")
        theoretical_type = "not_applicable"
    elif leaf.startswith("theoretical_"):
        family = "theoretical"
        empirical_method = "not_applicable"
        temporal_scope = "not_applicable"
        theoretical_type = leaf.removeprefix("theoretical_")
    else:
        family = "unclear"
        empirical_method = "unclear"
        temporal_scope = "unclear"
        theoretical_type = "unclear"

    try:
        confidence = float(item.get("confidence", 0))
    except (TypeError, ValueError):
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "article_id": clean_value(item.get("article_id")),
        "method_family_codex": family,
        "empirical_method_codex": empirical_method,
        "temporal_scope_codex": temporal_scope,
        "theoretical_type_codex": theoretical_type,
        "method_category_codex": leaf,
        "method_confidence_codex": confidence,
        "method_evidence_codex": clean_value(item.get("evidence")),
        "method_notes_codex": clean_value(item.get("notes")),
        "codex_model": CODEX_MODEL,
        "codex_reasoning_effort": CODEX_REASONING_EFFORT,
        "label_status": "codex_cli",
    }


def codex_command(schema_path, response_path):
    return [
        "codex",
        "exec",
        "--ephemeral",
        "--skip-git-repo-check",
        "--ask-for-approval",
        "never",
        "--sandbox",
        "read-only",
        "--cd",
        str(ROOT),
        "-m",
        CODEX_MODEL,
        "-c",
        f'model_reasoning_effort="{CODEX_REASONING_EFFORT}"',
        "--output-schema",
        str(schema_path),
        "--output-last-message",
        str(response_path),
        "-",
    ]


def classify_batch(records, batch_index):
    prompt = CODEX_CLASSIFICATION_PROMPT + "\n\nRecords to classify:\n" + json.dumps(records, ensure_ascii=False, indent=2)

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        schema_path = tmpdir / "method_schema.json"
        response_path = tmpdir / "codex_response.json"
        schema_path.write_text(json.dumps(CODEX_OUTPUT_SCHEMA, indent=2), encoding="utf-8")

        result = subprocess.run(
            codex_command(schema_path, response_path),
            input=prompt,
            text=True,
            capture_output=True,
            timeout=CODEX_TIMEOUT_SECONDS,
        )

        RAW_RESPONSE_DIR.mkdir(parents=True, exist_ok=True)
        raw_path = RAW_RESPONSE_DIR / f"{OUTPUT_SUFFIX}_batch_{batch_index:04d}.txt"
        raw_path.write_text(
            "STDOUT\n" + result.stdout + "\n\nSTDERR\n" + result.stderr,
            encoding="utf-8",
        )

        if result.returncode != 0:
            raise RuntimeError(f"Codex CLI failed for batch {batch_index}. See {raw_path}")

        response_text = response_path.read_text(encoding="utf-8") if response_path.exists() else result.stdout
        data = parse_json_object(response_text)
        classifications = data.get("classifications", [])
        normalized = [normalize_classification(item) for item in classifications]

    expected_ids = {record["article_id"] for record in records}
    returned_ids = {item["article_id"] for item in normalized}
    missing_ids = sorted(expected_ids - returned_ids)
    if missing_ids:
        raise RuntimeError(f"Codex response for batch {batch_index} missed article IDs: {missing_ids}")
    return normalized


target_df = select_target_articles(df)
records = [make_record(row) for _, row in target_df.iterrows()]
cached_by_id = load_cache(CACHE_PATH)
pending_records = [record for record in records if record["article_id"] not in cached_by_id]

print(f"Articles in corpus: {len(df)}")
print(f"Articles selected for this run: {len(records)}")
print(f"Already cached for selected articles: {len(records) - len(pending_records)}")
print(f"Pending Codex classifications: {len(pending_records)}")

new_classifications = []
for batch_start in range(0, len(pending_records), BATCH_SIZE):
    batch = pending_records[batch_start : batch_start + BATCH_SIZE]
    batch_index = batch_start // BATCH_SIZE + 1
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"Classifying batch {batch_index} ({len(batch)} articles), attempt {attempt}...")
            batch_classifications = classify_batch(batch, batch_index)
            append_cache(CACHE_PATH, batch_classifications)
            new_classifications.extend(batch_classifications)
            cached_by_id.update({item["article_id"]: item for item in batch_classifications})
            time.sleep(SLEEP_BETWEEN_CALLS)
            break
        except Exception as exc:
            if attempt >= MAX_RETRIES:
                raise
            print(f"Retrying batch {batch_index} after error: {exc}")
            time.sleep(5)

classification_rows = [cached_by_id.get(record["article_id"]) for record in records]
classification_rows = [row for row in classification_rows if row]
classifications = pd.DataFrame(classification_rows).drop_duplicates("article_id", keep="last")

labels = target_df[METADATA_COLUMNS].merge(classifications, on="article_id", how="left")
missing_mask = labels["method_category_codex"].isna()
if missing_mask.any():
    labels.loc[missing_mask, "method_family_codex"] = "unclear"
    labels.loc[missing_mask, "empirical_method_codex"] = "unclear"
    labels.loc[missing_mask, "temporal_scope_codex"] = "unclear"
    labels.loc[missing_mask, "theoretical_type_codex"] = "unclear"
    labels.loc[missing_mask, "method_category_codex"] = "unclear"
    labels.loc[missing_mask, "method_confidence_codex"] = 0.0
    labels.loc[missing_mask, "method_evidence_codex"] = ""
    labels.loc[missing_mask, "method_notes_codex"] = "missing Codex output"
    labels.loc[missing_mask, "codex_model"] = CODEX_MODEL
    labels.loc[missing_mask, "codex_reasoning_effort"] = CODEX_REASONING_EFFORT
    labels.loc[missing_mask, "label_status"] = "missing_codex_output"

write_csv(labels, LABELS_PATH)
if WRITE_COMPATIBILITY_OUTPUTS:
    write_csv(labels, ROOT / "outputs/tables/article_type_labels.csv")

display(labels.head())
display(labels["method_category_codex"].value_counts(dropna=False).rename_axis("method_category_codex").reset_index(name="n"))


In [ ]:
def validation_sample_for(labels_df):
    if labels_df.empty:
        return labels_df.copy()
    n = min(max(100, int(0.05 * len(labels_df))), len(labels_df))
    strata = labels_df[["period", "language", "method_category_codex"]].fillna("unknown").agg("|".join, axis=1)
    try:
        if strata.value_counts().min() >= 2 and strata.nunique() < n:
            sample, _ = train_test_split(labels_df, train_size=n, random_state=SAMPLE_RANDOM_STATE, stratify=strata)
            return sample.sort_values(["period", "method_category_codex", "year", "article_id"])
    except ValueError:
        pass
    return labels_df.sample(n=n, random_state=SAMPLE_RANDOM_STATE).sort_values(["period", "method_category_codex", "year", "article_id"])


validation_sample = validation_sample_for(labels)
validator_cols = ["validator_1_method_category", "validator_1_notes"]

if VALIDATION_PATH.exists():
    existing_validation = pd.read_csv(VALIDATION_PATH)
    reusable_cols = ["article_id"] + [col for col in validator_cols if col in existing_validation.columns]
    validation_sample = validation_sample.drop(columns=[col for col in validator_cols if col in validation_sample.columns], errors="ignore")
    validation_sample = validation_sample.merge(existing_validation[reusable_cols], on="article_id", how="left")

for col in validator_cols:
    if col not in validation_sample.columns:
        validation_sample[col] = ""
    validation_sample[col] = validation_sample[col].fillna("")

write_csv(validation_sample, VALIDATION_PATH)
if WRITE_COMPATIBILITY_OUTPUTS:
    write_csv(validation_sample, ROOT / "outputs/labels/article_type_validation_sample.csv")

results = []
checked = validation_sample[validation_sample["validator_1_method_category"].map(compact_ws).ne("")]
if len(checked):
    results.append({"metric": "n_validated_method_category", "value": len(checked)})
    results.append({
        "metric": "codex_human_agreement_method_category",
        "value": float((checked["validator_1_method_category"] == checked["method_category_codex"]).mean()),
    })
write_csv(pd.DataFrame(results, columns=["metric", "value"]), VALIDATION_RESULTS_PATH)
if WRITE_COMPATIBILITY_OUTPUTS:
    write_csv(pd.DataFrame(results, columns=["metric", "value"]), ROOT / "outputs/tables/article_type_validation_results.csv")

if not labels.empty:
    method_year = labels.groupby(["year", "method_category_codex"]).size().reset_index(name="n")
    method_year["share"] = method_year["n"] / method_year.groupby("year")["n"].transform("sum")

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.lineplot(data=method_year, x="year", y="share", hue="method_category_codex", ax=ax)
    scope_note = "sample" if RUN_SCOPE == "sample" else "full corpus"
    ax.set_title(f"Methodological category over time ({scope_note})")
    ax.set_xlabel("Year")
    ax.set_ylabel("Share")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Method category")
    fig.tight_layout()
    fig.savefig(FIGURE_PATH, dpi=220)
    save_caption(
        ROOT,
        FIGURE_PATH.name,
        "Codex CLI method-category labels over time using the empirical/theoretical taxonomy; sample runs are diagnostics only.",
    )
    if WRITE_COMPATIBILITY_OUTPUTS:
        fig.savefig(ROOT / "outputs/figures/fig_17_methodological_approach_over_time.png", dpi=220)
        save_caption(
            ROOT,
            "fig_17_methodological_approach_over_time.png",
            "Codex CLI method-category labels over time using the empirical/theoretical taxonomy.",
        )
    plt.show()

print(f"Validation sample: {VALIDATION_PATH}")
print(f"Validation results: {VALIDATION_RESULTS_PATH}")
print(f"Figure: {FIGURE_PATH}")
